# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup: connect to the FlyRank warehouse via DuckDB
import os
from pathlib import Path
import duckdb
from dotenv import load_dotenv

cwd = Path.cwd()
env_path = None
for parent in [cwd, *cwd.parents]:
    candidate = parent / ".env"
    if candidate.exists():
        env_path = candidate
        break

print("Current working directory:", cwd)
print(".env found at:", env_path if env_path else "NOT FOUND")

if env_path:
    load_dotenv(env_path, override=True)

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError(
        f"HF_TOKEN not found. Searched upward from {cwd} for a .env file."
    )

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
print("Connected to DuckDB + HF secret set.")


Current working directory: /run/media/zunkode/AIML_Studio/flyrank-ml-internship/work/notebooks
.env found at: /run/media/zunkode/AIML_Studio/flyrank-ml-internship/.env
Connected to DuckDB + HF secret set.


In [2]:
# Find the real path to the query_90d table instead of guessing.
# Lists every file in the repo, pulls out the ones for our lane, and builds
# LANE_TABLE from what's actually there.
from huggingface_hub import list_repo_files

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=hf_token)

print(f"Total files in repo: {len(files)}")
print()
top_level = sorted(set(f.split("/")[0] for f in files))
print("Top-level entries:", top_level)
print()

# Grab every file path that belongs to the query_90d table
matches = [f for f in files if "query_90d" in f]
print(f"Files matching 'query_90d': {len(matches)}")
for f in matches[:10]:
    print(" ", f)

if not matches:
    print()
    print("No match on 'query_90d' -- showing first 30 files in the repo instead,")
    print("so we can see the real naming convention:")
    for f in files[:30]:
        print(" ", f)
else:
    # Get the parent folder from the first match and build a glob from it
    parts = matches[0].split("/")
    # assuming the table lives in a folder named like the table itself
    table_folder = "/".join(parts[:-1]) if len(parts) > 1 else parts[0]
    LANE_TABLE = f"{BASE}/{table_folder}/**/*.parquet" if len(parts) > 1 else f"{BASE}/{parts[0]}"
    print()
    print("Derived LANE_TABLE =", LANE_TABLE)

    # Check it actually works
    result = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{LANE_TABLE}') LIMIT 1")
    print(result)


/home/zunkode/anaconda3/envs/CV/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total files in repo: 24

Top-level entries: ['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance', 'fact_content_daily_performance_sample.parquet', 'fact_content_query_90d.parquet']

Files matching 'query_90d': 1
  fact_content_query_90d.parquet

Derived LANE_TABLE = hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet
┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_hash_id                 │ VARCHAR     │ YES     │ NULL    │ NULL 

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane:** `fact_content_query_90d`. One pseudonymized client, one pseudonymized content item, one salted query hash, over a fixed 90-day window (`window_start` to `window_end`), with `last30`/`prev30` columns for trend.

- **One row =** one (`client_hash_id`, `content_hash_id`, `query_hash_id`) combination in the 90-day window. I checked it: 2,414,248 total rows against 133,852 distinct (client, content) pairs, so the query dimension really does split rows further.
- **Table(s) used:** `fact_content_query_90d` is my grain table. `dim_clients` and `dim_content` only come in for context and the availability check, never for the label.
- **Time window:** this table isn't partitioned by month like `fact_content_daily_performance`. It's one fixed 90-day snapshot as of the `v20260703` export, with `window_start`/`window_end` marking the span, and `last30`/`prev30` giving me a trend split inside it.
- **Predict/rank (label or proxy):** change in clicks, `clicks_last30` vs `clicks_prev30`, as a directional trend. Not a guaranteed causal outcome, just what moved.
- **Deliberately excluded:** `fact_content_daily_performance`. Different grain (daily, no query dimension), so mixing it in here would break my contract's grain.


In [5]:
# Check the grain in code instead of just claiming it in prose
con.sql(f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{LANE_TABLE}')
""".format(LANE_TABLE=LANE_TABLE))


┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│    2414248 │
└────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Field | Why |
|---|---|---|
| Feature | `clicks_prev30` | already a closed period, known at decision time |
| Feature | `avg_position_last30` | observed ranking so far |
| Feature | `content_total_impressions_90d` | cumulative demand signal over the full window |
| Feature | `rare_impressions_share` | observed long-tail query mix |
| Feature | `query_token_count` | static property of the query text |
| Label | `clicks_last30` (used to get last30 minus prev30) | the outcome I'm predicting |
| Context | `client_hash_id`, `content_hash_id`, `query_hash_id` | identify the row, not predictive on their own |
| Excluded | `anonymized_impressions_share` | Google-anonymized traffic is aggregated noise at this grain, not a real per-row signal |


In [6]:
# List every column so I can sort them into the buckets above
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{LANE_TABLE}')")


┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_hash_id                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_char_count              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ query_token_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ window_start                  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ window_end                    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ impressions_90d               

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries: grain check, row count and date span, then availability with `IS TRUE`. Feature frame and the leakage trap come after.

In [7]:
# Query 1: grain check. One row should be one (client, content, query) combo
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (client_hash_id, content_hash_id, query_hash_id)) AS distinct_combos
    FROM read_parquet('{LANE_TABLE}')
""")
# total_rows should match distinct_combos. If it does, my grain claim in Section 1 holds.


┌────────────┬─────────────────────────┐
│ total_rows │ distinct_client_content │
│   int64    │          int64          │
├────────────┼─────────────────────────┤
│    2414248 │                  133852 │
└────────────┴─────────────────────────┘

In [8]:
# Query 2: row count and real date span.
# No month partition on this table, so my slice is the whole fixed window,
# bounded by window_start and window_end.
con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(window_start) AS earliest_start,
           MAX(window_end) AS latest_end
    FROM read_parquet('{LANE_TABLE}')
""")


┌─────────┐
│ n_rows  │
│  int64  │
├─────────┤
│ 2414248 │
└─────────┘

In [9]:
# Query 3: availability filter with IS TRUE.
# No boolean column on the fact table, so I'm checking the dimension tables instead.
DIM_CLIENTS = f"{BASE}/dim_clients.parquet"
DIM_CONTENT = f"{BASE}/dim_content.parquet"

dim_clients_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DIM_CLIENTS}')").df()
dim_content_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DIM_CONTENT}')").df()

bool_in_clients = dim_clients_cols[dim_clients_cols['column_type'] == 'BOOLEAN']['column_name'].tolist()
bool_in_content = dim_content_cols[dim_content_cols['column_type'] == 'BOOLEAN']['column_name'].tolist()

print("Boolean columns, dim_clients:", bool_in_clients, "| dim_content:", bool_in_content)
print()

# Run the actual IS TRUE filter against whichever boolean column turns up first.
# I'm preferring dim_content since this lane is content-grained.
if bool_in_content:
    bool_col = bool_in_content[0]
    result = con.sql(f"""
        SELECT COUNT(*) AS available_rows
        FROM read_parquet('{LANE_TABLE}') f
        JOIN read_parquet('{DIM_CONTENT}') c ON f.content_hash_id = c.content_hash_id
        WHERE c.{bool_col} IS TRUE
    """)
    print(f"Filtered on dim_content.{bool_col} IS TRUE:")
    print(result)
elif bool_in_clients:
    bool_col = bool_in_clients[0]
    result = con.sql(f"""
        SELECT COUNT(*) AS available_rows
        FROM read_parquet('{LANE_TABLE}') f
        JOIN read_parquet('{DIM_CLIENTS}') cl ON f.client_hash_id = cl.client_hash_id
        WHERE cl.{bool_col} IS TRUE
    """)
    print(f"Filtered on dim_clients.{bool_col} IS TRUE:")
    print(result)
else:
    print("No boolean column found in either dimension table.")
    print("Availability isn't expressible as an IS TRUE filter for this lane,")
    print("and that itself is worth noting as a limitation in Section 4.")


ParserException: Parser Error: syntax error at or near "<"

LINE 4:     WHERE <BOOLEAN_COLUMN_HERE> IS TRUE
                  ^

In [10]:
# 5-feature frame, max 5. Each comment states when it's available and why
# it's knowable at the decision moment.
features_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        clicks_prev30,
        avg_position_last30,
        content_total_impressions_90d,
        rare_impressions_share,
        query_token_count
    FROM read_parquet('{LANE_TABLE}')
    LIMIT 1000
""").df()

# clicks_prev30: available once the prev30 window closes, knowable at decision
# time because it's fully historical.
# avg_position_last30: available once the last30 window closes, knowable at
# decision time because it's an observed outcome, not a forecast.
# content_total_impressions_90d: available once the full 90d window closes,
# knowable at decision time because it's a cumulative count of what already happened.
# rare_impressions_share: available from the same closed 90d window, knowable at
# decision time because it's an observed mix, not predicted.
# query_token_count: always available, it's a static property of the query text,
# knowable at decision time because it never changes once the query exists.

features_df.head()


,client_hash_id,content_hash_id,clicks_last30,avg_position_last30,content_total_impressions_90d,rare_impressions_share,clicks_prev30
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,0,NaN,1466,0.043656,0
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,0,NaN,1466,0.043656,0
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,0,24.272727,1466,0.043656,0
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,0,13.000000,1466,0.043656,0
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,0,NaN,1466,0.043656,0


In [11]:
# The trap: I add one label-derived column on purpose, watch the score jump,
# then take it out and keep the honest number.
# Simple label: did clicks go up last30 vs prev30. 1 for yes, 0 for no.
labeled = con.sql(f"""
    SELECT
        clicks_prev30, avg_position_last30, content_total_impressions_90d,
        rare_impressions_share, query_token_count,
        clicks_last30, clicks_90d,
        CASE WHEN clicks_last30 > clicks_prev30 THEN 1 ELSE 0 END AS label_went_up
    FROM read_parquet('{LANE_TABLE}')
    WHERE clicks_prev30 IS NOT NULL AND clicks_last30 IS NOT NULL
    LIMIT 20000
""").df()

honest_features = ['clicks_prev30', 'avg_position_last30', 'content_total_impressions_90d',
                    'rare_impressions_share', 'query_token_count']

# Leaky version: clicks_90d is literally clicks_last30 plus clicks_prev30. It's
# built straight from the label's own inputs, so the quick score looks fake-good.
leaky_features = honest_features + ['clicks_90d']

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = labeled[leaky_features].fillna(0)
y = labeled['label_went_up']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
leaky_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("With the leaky clicks_90d column, AUC:", round(leaky_auc, 4))

# Take the leaky column out and re-check. This is the honest number.
X_honest = labeled[honest_features].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Without it, honest AUC:", round(honest_auc, 4))

print()
print(f"Leakage inflated AUC by {round(leaky_auc - honest_auc, 4)}. clicks_90d is just")
print("clicks_last30 plus clicks_prev30, so it directly encodes the label. Took it out above.")


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced panel:** per-client history depth differs (`dim_clients.gsc_data_start` / `ga4_data_start`), so early rows for newer clients can be thinner or GSC-only.
- **Salted hashes:** query and content hash keys are salted and namespaced per release. They won't match a future export by key, so nothing here joins across releases.
- **Rare-tail aggregation:** queries under 10 impressions and Google-anonymized impressions get folded into per-content aggregate shares. Long-tail query behavior is only knowable in aggregate, never row by row.
- This table has no per-row availability flag of its own. Any "is this row usable" check has to go through a join to `dim_clients` or `dim_content`. That's a real constraint on this lane, not just a modeling choice.


In [12]:
# Backing up the unbalanced panel claim with an actual query
con.sql(f"""
    SELECT
        MIN(gsc_data_start) AS earliest_gsc_start,
        MAX(gsc_data_start) AS latest_gsc_start,
        MIN(ga4_data_start) AS earliest_ga4_start,
        MAX(ga4_data_start) AS latest_ga4_start
    FROM read_parquet('{DIM_CLIENTS}')
""")
# A wide spread between earliest and latest start dates backs up the unbalanced
# panel claim above. Some clients just have more history than others.

# Second limitation, backed by data. How much of this table is rare or anonymized tail?
con.sql(f"""
    SELECT
        AVG(rare_impressions_share) AS avg_rare_share,
        AVG(anonymized_impressions_share) AS avg_anonymized_share
    FROM read_parquet('{LANE_TABLE}')
""")
# High averages here back up the rare-tail aggregation limitation, straight from the data.


HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/dim_clients' (HTTP 404)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.